In [10]:
import os
import psycopg2
import pandas as pd
import warnings
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

warnings.filterwarnings("ignore")


os.environ["SPOTIPY_CLIENT_ID"] = "c97d261239c2476d9b85f08179450fc1"
os.environ["SPOTIPY_CLIENT_SECRET"] = "9262159ff2264fefa2568620aab11d97"

auth_manager = SpotifyClientCredentials()
sp = spotipy.Spotify(auth_manager=auth_manager)


def get_artist_data(artist_name):
    results = sp.search(q=f"artist:{artist_name}", type="artist")
    items = results['artists']['items']
    data = []
    for artist in items:
        data.append({
            "artist_name": artist['name'],
            "artist_id": artist['id'],
            "artist_uri": artist['uri'],
            "followers": artist['followers']['total'],
            "genres": ", ".join(artist['genres']),
            "popularity": artist['popularity'],
            "image_url": artist['images'][0]['url'] if artist['images'] else None
        })
    return pd.DataFrame(data)

artist_df = get_artist_data("Arijit Singh")   


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",   
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "spotify_artists"
columns = artist_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

# Drop if exists (avoid schema mismatch), then recreate
cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()

print("Table recreated with schema:")
print(create_stmt)


columns_list = list(artist_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f"""
INSERT INTO "{table_name}" ({quoted_cols})
VALUES ({placeholders})
"""

for _, row in artist_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)

cur.close()
conn.close()

df_from_db


Table recreated with schema:

CREATE TABLE "spotify_artists" (
  "artist_name" TEXT,
  "artist_id" TEXT,
  "artist_uri" TEXT,
  "followers" INT,
  "genres" TEXT,
  "popularity" INT,
  "image_url" TEXT
);

Data inserted


,artist_name,artist_id,artist_uri,followers,genres,popularity,image_url
0,Arijit Singh,4YRxDV8wJFPHPTeXepOstw,spotify:artist:4YRxDV8wJFPHPTeXepOstw,157248833,"hindi pop, bollywood, desi, bangla pop",93,https://i.scdn.co/image/ab6761610000e5eb5ba2d7...
1,Arijit Singh,6zrQkGA8EvD1Y4z35lIc2e,spotify:artist:6zrQkGA8EvD1Y4z35lIc2e,3404,"bangla pop, bhajan, devotional",27,https://i.scdn.co/image/ab67616d0000b273fff068...
2,"Hariharan, Swarnalatha, Kumar Sanu, Sapna Mukh...",2b84vRVbe4QTox0pMqAC6W,spotify:artist:2b84vRVbe4QTox0pMqAC6W,998,,10,https://i.scdn.co/image/ab67616d0000b2737bbcbc...
3,"Vishal-Shekhar,Arijit Singh,Shilpa Rao,Kumaar",0l1GEW74dNahpWUZ6VY2aG,spotify:artist:0l1GEW74dNahpWUZ6VY2aG,38,,0,None
4,Arijit Singh,11P6oIAn8olEk5FJSbhAUy,spotify:artist:11P6oIAn8olEk5FJSbhAUy,20,,0,https://i.scdn.co/image/ab67616d0000b273bf7005...
5,Arijit Singh,1Y7GoYOL4iwZNPULQbKqkF,spotify:artist:1Y7GoYOL4iwZNPULQbKqkF,204,,0,https://i.scdn.co/image/ab67616d0000b2733a28ea...
6,"Sachin-Jigar,Arijit Singh,Neeti Mohan,Vayu",4hMvXl2N76KCluwwzziGpq,spotify:artist:4hMvXl2N76KCluwwzziGpq,14,,0,None
7,Arijitt singh,63tnhrqJroQi6keC7fU1rQ,spotify:artist:63tnhrqJroQi6keC7fU1rQ,30,,0,https://i.scdn.co/image/ab67616d0000b27344ce83...
8,"Vishal-Shekhar,Arijit Singh,Jaideep Sahni",5g7Oo0qcaRJpXaSTVwdRv0,spotify:artist:5g7Oo0qcaRJpXaSTVwdRv0,6,,0,None
9,"Sachin-Jigar,Arijit Singh,Priya Saraiya",69BzYn1D0VM66nWt01CwlQ,spotify:artist:69BzYn1D0VM66nWt01CwlQ,77,,0,None
